# Gemma GSPO Evaluation
Tests the Gemma GSPO model using the correct chat template and scores dialect density on outputs.
Compares against the SFT baseline to check for dialect improvement.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
GSPO_MODEL_ID = "jordanpainter/dialect-gemma-gspo-all"
SFT_MODEL_ID  = "jordanpainter/DialLM-Gemma-sft-all"
TOKENIZER_ID  = "google/gemma-3-4b-it"   # instruct tokenizer with correct chat template

DIALECT_MODEL = "srirag/feature-identifier"
SYSTEM_PROMPT = "You are a helpful assistant."
MAX_NEW_TOKENS = 256
LOAD_IN_4BIT = True   # set False if you have enough VRAM

In [ ]:
# ── Prompts ──────────────────────────────────────────────────────────────────
# Mix of general prompts and ones likely to elicit dialectal responses
PROMPTS = [
    # General
    "Describe your morning routine in a casual, conversational way.",
    "Give me some advice on learning a new skill.",
    "Explain what machine learning is, like you're chatting with a friend.",
    # Dialect-eliciting
    "What do you reckon is the best way to spend a Sunday afternoon?",
    "Tell me about a time you were well chuffed about something.",
    "Write a short message to a mate about plans for the weekend.",
    "What's the craic? Tell me something interesting about language.",
    "I'm feeling a bit knackered today. Any tips for getting your energy back?",
    "What do you make of the weather lately? Been absolutely minging hasn't it.",
    "Give me your honest opinion — is it worth learning to cook proper homemade food?",
]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

def load_model(model_id):
    print(f"\nLoading {model_id} ({'4-bit' if LOAD_IN_4BIT else 'bf16'})...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    kwargs = dict(device_map="auto", low_cpu_mem_usage=True)
    if LOAD_IN_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()

    # Vocab sanity check
    embed_rows = model.get_input_embeddings().weight.shape[0]
    tok_len = len(tokenizer)
    print(f"  tokenizer vocab: {tok_len} | embedding rows: {embed_rows}")
    if tok_len > embed_rows:
        raise ValueError(f"Tokenizer vocab ({tok_len}) > embedding rows ({embed_rows}) — wrong tokenizer")

    # Debug: print EOS token info
    print(f"  eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
    eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    print(f"  <end_of_turn> id: {eot_id}")

    return model, tokenizer


def build_prompt(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, prompt):
    built = build_prompt(tokenizer, prompt)
    inputs = tokenizer(built, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Only use the tokenizer eos — don't add <end_of_turn> as EOS in case it's firing immediately
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id else []

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=10,           # force at least 10 tokens
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    # Debug: show first few token ids
    print(f"  first token ids: {new_tokens[0][:10].tolist()}")
    return tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()

In [ ]:
# ── Dialect scorer ───────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from rewards.dialect_reward_model import DialectDensityScorer

device = "cuda" if torch.cuda.is_available() else "cpu"
scorer = DialectDensityScorer(model_path=DIALECT_MODEL, device=device)

def dialect_score(texts):
    scores = scorer.score_density(texts)
    if isinstance(scores, torch.Tensor):
        scores = scores.detach().cpu().tolist()
    return [round(float(s), 4) for s in scores]

In [ ]:
# ── Run both models ──────────────────────────────────────────────────────────
import pandas as pd

results = []

for label, model_id in [("GSPO", GSPO_MODEL_ID), ("SFT", SFT_MODEL_ID)]:
    model, tokenizer = load_model(model_id)
    for prompt in PROMPTS:
        response = generate(model, tokenizer, prompt)
        density = dialect_score([response])[0]
        results.append({"model": label, "prompt": prompt, "response": response, "dialect_density": density})
        print(f"[{label}] {density:.4f} | {response[:80]}")
    del model
    torch.cuda.empty_cache()

df = pd.DataFrame(results)

In [ ]:
# ── Summary: GSPO vs SFT ─────────────────────────────────────────────────────
summary = df.groupby("model")["dialect_density"].agg(["mean", "std", "min", "max"]).round(4)
print(summary)

gspo_mean = df[df["model"] == "GSPO"]["dialect_density"].mean()
sft_mean  = df[df["model"] == "SFT"]["dialect_density"].mean()
print(f"\nGSPO improvement over SFT: {gspo_mean - sft_mean:+.4f}")

In [ ]:
# ── Per-prompt side-by-side ───────────────────────────────────────────────────
for prompt in PROMPTS:
    sub = df[df["prompt"] == prompt]
    gspo = sub[sub["model"] == "GSPO"].iloc[0]
    sft  = sub[sub["model"] == "SFT"].iloc[0]
    delta = gspo["dialect_density"] - sft["dialect_density"]
    print(f"\nPrompt: {prompt}")
    print(f"  SFT  ({sft['dialect_density']:.4f}): {sft['response'][:200]}")
    print(f"  GSPO ({gspo['dialect_density']:.4f}): {gspo['response'][:200]}")
    print(f"  Δ = {delta:+.4f}")

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv("gemma_gspo_vs_sft.csv", index=False)
print("Saved to gemma_gspo_vs_sft.csv")